# Laboratorio: subespacios, bases, núcleo e imagen

Este laboratorio usa aritmética simbólica exacta. El objetivo es obtener bases y dimensiones, pero también verificar por qué los vectores elegidos cumplen las condiciones requeridas.

Trabajaremos con:

1. generadores colocados como columnas;
2. subespacios definidos por ecuaciones homogéneas;
3. coordenadas respecto de una base;
4. núcleo, imagen, espacio fila y rango–nulidad;
5. sistemas compatibles e incompatibles.

In [ ]:
import sympy as sp

sp.init_printing()

## 1. Funciones de apoyo

La función siguiente aplica las reglas teóricas:

- base de la imagen: columnas pivote de la matriz original;
- base del espacio fila: filas no nulas de la RREF;
- base del núcleo: vectores de la solución homogénea;
- rango: número de pivotes.

In [ ]:
def espacios_fundamentales(A):
    A = sp.Matrix(A)
    R, pivotes = A.rref()
    base_imagen = [A[:, j] for j in pivotes]
    base_fila = [sp.Matrix(1, A.cols, list(R.row(i)))
                 for i in range(R.rows) if any(R.row(i))]
    base_nucleo = A.nullspace()
    return {
        "A": A,
        "R": R,
        "pivotes": pivotes,
        "rango": len(pivotes),
        "nulidad": len(base_nucleo),
        "base_imagen": base_imagen,
        "base_fila": base_fila,
        "base_nucleo": base_nucleo,
    }


def verificar_base_nucleo(A, base):
    A = sp.Matrix(A)
    return all(A * v == sp.zeros(A.rows, 1) for v in base)

## 2. Reducir un conjunto generador

Consideremos

$$
W=\operatorname{span}\{(1,-1,1),(2,0,1),(-4,-2,-1),(3,1,2)\}.
$$

Los vectores se colocan como columnas. Los índices de pivote que devuelve Python comienzan en cero.

In [ ]:
generadores = [
    sp.Matrix([1, -1, 1]),
    sp.Matrix([2, 0, 1]),
    sp.Matrix([-4, -2, -1]),
    sp.Matrix([3, 1, 2]),
]
G = sp.Matrix.hstack(*generadores)
datos_G = espacios_fundamentales(G)

display(G, datos_G["R"])
print("columnas pivote:", tuple(j + 1 for j in datos_G["pivotes"]))
print("dimensión de W:", datos_G["rango"])
print("base tomada de los generadores originales:")
for v in datos_G["base_imagen"]:
    display(v)

assert datos_G["pivotes"] == (0, 1, 3)
assert generadores[2] == 2 * generadores[0] - 3 * generadores[1]

## 3. Subespacio definido por ecuaciones

Sea

$$
W=\left\{x\in\mathbb R^4:
\begin{aligned}
x_1+2x_2-4x_3+3x_4&=0,\\
-x_1-2x_3+x_4&=0,\\
x_1+x_2-x_3+2x_4&=0
\end{aligned}\right\}.
$$

Este conjunto es el núcleo de la matriz de coeficientes. Una base se obtiene al describir todas las soluciones del sistema homogéneo.

In [ ]:
A_W = sp.Matrix([
    [1, 2, -4, 3],
    [-1, 0, -2, 1],
    [1, 1, -1, 2],
])
datos_W = espacios_fundamentales(A_W)

display(A_W, datos_W["R"])
print("rango:", datos_W["rango"])
print("dimensión de W = nulidad:", datos_W["nulidad"])
print("base de W:")
for v in datos_W["base_nucleo"]:
    display(v)

assert datos_W["rango"] == 3
assert datos_W["nulidad"] == 1
assert verificar_base_nucleo(A_W, datos_W["base_nucleo"])

## 4. Coordenadas respecto de una base

Si las columnas de $B$ forman una base, las coordenadas de $x$ se obtienen resolviendo

$$
B[x]_{\mathcal B}=x.
$$

La reconstrucción $B[x]_{\mathcal B}=x$ es una verificación obligatoria.

In [ ]:
def coordenadas_en_base(x, vectores_base):
    B = sp.Matrix.hstack(*[sp.Matrix(v) for v in vectores_base])
    x = sp.Matrix(x)
    if B.rows != B.cols or B.det() == 0:
        raise ValueError("Los vectores deben formar una base de todo R^n")
    coordenadas = B.LUsolve(x)
    return B, coordenadas


base_B = [sp.Matrix([1, 1]), sp.Matrix([1, -1])]
x = sp.Matrix([5, 1])
B, x_B = coordenadas_en_base(x, base_B)

print("matriz de la base:")
display(B)
print("coordenadas [x]_B:")
display(x_B)
print("reconstrucción:")
display(B * x_B)
assert B * x_B == x

## 5. Ejemplo de profundidad PC1

Analizamos una matriz de $6\times8$. El tamaño obliga a organizar el procedimiento: RREF, pivotes, base de la imagen, base del núcleo y verificación de rango–nulidad.

In [ ]:
M = sp.Matrix([
    [1, 2, 0, 1, 3, 0, 1, 4],
    [0, 1, 1, 2, 1, 0, 0, 3],
    [1, 3, 1, 3, 4, 0, 1, 7],
    [2, 5, 1, 4, 7, 0, 1, 10],
    [0, 0, 0, 0, 0, 1, 1, 0],
    [1, 1, 0, 0, 2, 1, 0, 2],
])
datos_M = espacios_fundamentales(M)

display(M, datos_M["R"])
print("pivotes:", tuple(j + 1 for j in datos_M["pivotes"]))
print("rango:", datos_M["rango"])
print("nulidad:", datos_M["nulidad"])
print("base de la imagen:")
for v in datos_M["base_imagen"]:
    display(v)
print("base del núcleo:")
for v in datos_M["base_nucleo"]:
    display(v)

assert datos_M["rango"] == 5
assert datos_M["nulidad"] == 3
assert datos_M["rango"] + datos_M["nulidad"] == M.cols
assert verificar_base_nucleo(M, datos_M["base_nucleo"])
assert sp.Matrix.hstack(*datos_M["base_imagen"]).rank() == datos_M["rango"]

## 6. Compatibilidad y solución afín

Para el vector

$$
b=(1,0,2,1,-1,0)^T,
$$

la columna aumentada crea un pivote adicional, de modo que $b\notin\operatorname{Im}(M)$ y el sistema es incompatible.

Después construiremos un sistema compatible eligiendo primero $x_p$ y definiendo $b_0=Mx_p$. Todas sus soluciones serán $x_p+\ker(M)$.

In [ ]:
b = sp.Matrix([1, 0, 2, 1, -1, 0])
print("rango(M) =", M.rank())
print("rango([M|b]) =", M.row_join(b).rank())
assert M.rank() == 5 and M.row_join(b).rank() == 6
assert sp.linsolve((M, b)) is sp.EmptySet

x_particular = sp.Matrix([1, 0, -1, 2, 0, 1, 0, -2])
b0 = M * x_particular
print("un término independiente compatible b0:")
display(b0)

for z in datos_M["base_nucleo"]:
    assert M * (x_particular + z) == b0

print("Cada x_particular + z, con z en ker(M), vuelve a producir b0.")

## 7. Ejercicios de laboratorio

1. Calcule bases del espacio fila, la imagen y el núcleo de

   $$
   \begin{bmatrix}1&2&3&4\\2&4&6&8\\1&1&1&1\end{bmatrix}.
   $$

2. Modifique una entrada de la matriz grande $M$ y determine qué cambia en rango, nulidad y posiciones pivote.
3. Para el sistema compatible $Mx=b_0$, escriba la solución completa usando tres parámetros y la base calculada del núcleo.
4. Construya dos subespacios $U,W\subseteq\mathbb R^4$ mediante generadores. Calcule una base de $U+W$ colocando juntos todos los generadores.
5. Elabore un contraejemplo computacional que muestre que las columnas no nulas de la RREF no deben usarse como base de la imagen original.

```{admonition} Criterio de entrega
:class: tip
Además de las salidas de Python, indique qué vectores forman cada base, su dimensión y la verificación algebraica correspondiente.
```